# 03 — Normalization

Walks the real provider-payload → Stage 3 domain-model boundary. Provider records never enter the feature/label/model pipeline directly: every field passes through a `normalize_*` function here and nowhere else.

**Data mode: REAL DATA.** Every figure below is computed from the genuine Bloomberg CSV exports for this strategy, through production AtlasQuant code. No synthetic, simulated, imputed, or approximated value appears anywhere in this notebook.

Those exports are gitignored, so they are absent in a fresh clone. When they are missing this notebook still executes top to bottom, but every computing cell prints an explicit *real data not present — cannot produce a genuine result* notice instead of a number. It never falls back to synthetic data: a plausible-looking figure derived from invented inputs is not a result (see `CLAUDE.md`).

In [ ]:
import _real_data as rd

print(rd.banner())
AVAILABLE = rd.data_available()


## One real fundamentals row, before and after

`RawFundamentalsRow` is provider-shaped input; `FundamentalsFeatureRecord` is the Stage 3 type. Note that `features` is an open bag of already-derived named features — this strategy's provider delivers those directly, unlike a fixed set of raw filing line items.

In [ ]:
if AVAILABLE:
    from atlas_quant.strategies.multi_factor_ranking_ml.acquisition.fundamentals_quarterly import (
        read_fundamentals_quarterly,
    )
    from atlas_quant.strategies.multi_factor_ranking_ml.production.normalization import (
        normalize_fundamentals_row,
    )

    raw_rows = read_fundamentals_quarterly(rd.RAW_ROOT / "fundamentals_quarterly.csv")
    sample_raw = next(r for r in raw_rows if r.quarter_end.year == 2023 and r.gics_sector)

    print("RawFundamentalsRow")
    print("  symbol      ", sample_raw.symbol)
    print("  asset_class ", sample_raw.asset_class)
    print("  fiscal      ", sample_raw.fiscal_period, sample_raw.fiscal_year)
    print("  quarter_end ", sample_raw.quarter_end)
    print("  filed_at    ", sample_raw.filed_at, "(estimated)" if sample_raw.filed_at_is_estimated else "(observed)")
    print("  gics_sector ", sample_raw.gics_sector)
    print("  features    ", len(sample_raw.features), "keys")

    record = normalize_fundamentals_row(sample_raw)
    print()
    print("FundamentalsFeatureRecord")
    print("  instrument_id", record.instrument_id)
    print("  quarter_end  ", record.quarter_end)
    print("  filed_at     ", record.filed_at)
    print("  gics_sector  ", record.gics_sector)
    print("  provenance   ", record.provenance)
else:
    raw_rows = ()
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Batch normalization

A record that fails to normalize is dropped from the batch and reported as an `ERROR`-severity `DataValidationIssue`. It never crashes the run, and never silently becomes a default value.

In [ ]:
if AVAILABLE:
    from atlas_quant.strategies.multi_factor_ranking_ml.production.normalization import (
        normalize_fundamentals_batch,
    )

    records, issues = normalize_fundamentals_batch(raw_rows)
    print(f"in  {len(raw_rows):,} raw rows")
    print(f"out {len(records):,} FundamentalsFeatureRecord  |  {len(issues):,} rejected")
    for issue in issues[:5]:
        print(f"  {issue.severity.value:<8} {issue.subject:<10} {issue.message[:100]}")
else:
    records = ()
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Universe membership

The universe is derived from the fundamentals export itself — there is no separate index-membership source, which is exactly why it is survivorship-biased and says so.

In [ ]:
if AVAILABLE:
    from datetime import datetime

    from atlas_quant.strategies.multi_factor_ranking_ml.acquisition.universe import (
        universe_records_from_fundamentals,
    )
    from atlas_quant.strategies.multi_factor_ranking_ml.production.normalization import (
        normalize_universe,
    )

    universe_raw = universe_records_from_fundamentals(raw_rows, retrieved_at=datetime.now())
    members, universe_issues = normalize_universe(universe_raw)
    print(f"{len(members):,} universe members  |  {len(universe_issues)} rejected")
    print("survivorship_biased:", {m.survivorship_biased for m in members})
    print("source             :", {m.source for m in members})
    print("examples           :", [m.instrument_id.symbol for m in members[:8]])
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Sector facts

`sector_records_from_fundamentals` derives one `SectorRecord` per fundamentals record, `as_of=record.filed_at`. This is this strategy's *only* sector source — a present-day GICS classification applied to every historical row for a ticker. That is a disclosed caveat, not a fixed one.

In [ ]:
if AVAILABLE:
    from collections import Counter

    from atlas_quant.strategies.multi_factor_ranking_ml.production.normalization import (
        sector_records_from_fundamentals,
    )

    sector_records = sector_records_from_fundamentals(records)
    latest_sector = {}
    for sector_record in sector_records:
        latest_sector[sector_record.instrument_id] = sector_record.raw_sector

    print(f"{len(sector_records):,} SectorRecord facts over {len(latest_sector):,} instruments")
    print()
    for sector, count in Counter(latest_sector.values()).most_common():
        print(f"  {str(sector):<26} {count:>5,}")

    print()
    print("Materials is the excluded sector (config.exclude_sectors):", rd.config().exclude_sectors)
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Prices

`read_close_price` requires an explicit `price_convention`: it is not inferrable from the file, and asserting the wrong one would silently corrupt every forward return across a split. This pull is `split_dividend_adjusted`, confirmed empirically against known splits.

In [ ]:
if AVAILABLE:
    from atlas_quant.strategies.multi_factor_ranking_ml.acquisition.close_price import read_close_price
    from atlas_quant.strategies.multi_factor_ranking_ml.production.normalization import normalize_prices

    raw_prices = read_close_price(
        rd.RAW_ROOT / "Close_Price.csv", price_convention=rd.PRICE_CONVENTION
    )
    print(f"{len(raw_prices):,} RawPriceRecord parsed (blank cells omitted, never zero-filled)")

    # Normalize a bounded head slice: normalizing all 9.2M rows takes ~20s and
    # demonstrates nothing the first few thousand do not.
    observations, price_issues = normalize_prices(raw_prices[:5000])
    print(f"normalized head slice: {len(observations):,} DailyPriceObservation, "
          f"{len(price_issues)} rejected")
    print()
    print("example:", observations[0])
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Findings

- The full real fundamentals export normalizes cleanly into Stage 3 types; the handful of rejections are reported as structured validation issues rather than swallowed.
- Sector comes from the fundamentals rows themselves, one `SectorRecord` per row at `as_of=filed_at`. Materials — the one excluded sector — is a real, populated sector in this universe.
- The price convention is an explicit caller assertion at the parse boundary, not an inference.

## Limitations

- GICS sector is a present-day snapshot applied backwards over each ticker's whole history. It is therefore **not genuinely point-in-time**: a company that changed sector classification is treated as always having had its current one. Classified `data_provenance_required`.
- Universe membership is derived from the export's ticker list, so it is survivorship-biased by construction (`survivorship_biased=True` is recorded on every membership record rather than hidden).
- Only a head slice of prices is normalized here, for runtime. The full normalization runs in the backtest path (notebooks 08–09).